# 04 · Preprocessing

Build the preprocessing pipeline and **fit it on train only**. V1–V28 are already PCA-scaled and pass through untouched; only `Amount` and `Time` are transformed. For class imbalance we use **class weights / `scale_pos_weight`**, never global undersampling, and we evaluate on the natural distribution.

In [1]:
# --- project-root bootstrap: portable across VS Code / Jupyter / CLI ---
import os
from pathlib import Path
_p = Path.cwd()
while not (_p / 'config' / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
os.chdir(_p)
print('working dir:', Path.cwd())

working dir: /Users/asfalanoi/app_2026/fraud_detection


In [2]:
from fraud.io import read_parquet
from fraud.preprocess import build_preprocessor

train = read_parquet('data/processed/train.parquet')
valid = read_parquet('data/processed/valid.parquet')
FEATURES = ['Time', 'Amount'] + [f'V{i}' for i in range(1, 29)]
X_train, y_train = train[FEATURES], train['Class']
X_train.shape

(227845, 30)

In [3]:
# Fit on TRAIN ONLY — no leakage
pre = build_preprocessor()
X_train_t = pre.fit_transform(X_train)
print('transformed shape:', X_train_t.shape)
# columns: 28 V passthrough + 1 log-amount + 1 robust-amount + 2 cyclical-hour = 32

transformed shape: (227845, 32)


In [4]:
# Imbalance strategy (not undersampling)
neg, pos = int((y_train == 0).sum()), int((y_train == 1).sum())
print(f'negatives={neg:,}  positives={pos:,}  ratio={neg/pos:.1f}:1')
print('scale_pos_weight for XGBoost:', round(neg / pos, 2))

negatives=227,451  positives=394  ratio=577.3:1
scale_pos_weight for XGBoost: 577.29


**Why not undersample globally?** It throws away ~99.8% of legitimate data and, if done before the split, leaks information. Class weights keep all data and let the model see the true distribution. **Next:** `05_feature_engineering`.